# CSoT'26 - ML in Astronomy - Week 3 . Part 1: A CNN and the Training Loop (Solution)

Reference implementation. **Only open after attempting [`week3_cnn_starter.ipynb`](week3_cnn_starter.ipynb).**

Companion reading: [`01-convolutions-and-pooling.md`](../01-convolutions-and-pooling.md), [`02-building-a-cnn.md`](../02-building-a-cnn.md), [`03-the-training-loop.md`](../03-the-training-loop.md).

## Step 0 - Re-create the Week 1 data pipeline

Reproduced from [`week1_data_solution.ipynb`](../../Week-1/notebooks/week1_data_solution.ipynb). The download is commented out - uncomment it the first time you run on a fresh Colab. If you saved `galaxy_data/` to Drive in Week 1, re-mount Drive and point `DATA_ROOT` at it.

In [ ]:
import os
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

In [ ]:
RAW_ROOT = Path("galaxy_raw")
IMAGES_DIR = RAW_ROOT / "images_gz2"   # adjust if your JPGs landed one folder deeper
DATA_ROOT = Path("galaxy_data")        # train/val/test subfolders get created here
LABELS_URL = "https://gz2hart.s3.amazonaws.com/gz2_hart16.csv.gz"

# --- Download via Kaggle API (run once) ---
# from google.colab import files
# files.upload()  # select kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip -q install kaggle pandas
# !kaggle datasets download -d jaimetrickz/galaxy-zoo-2-images
# !unzip -q -o galaxy-zoo-2-images.zip -d {RAW_ROOT}
# !unzip -q -o {RAW_ROOT}/images_gz2.zip -d {IMAGES_DIR}
# !wget -q -O {RAW_ROOT}/gz2_hart16.csv.gz {LABELS_URL}
# !gunzip -f {RAW_ROOT}/gz2_hart16.csv.gz

print("RAW_ROOT   =", RAW_ROOT)
print("IMAGES_DIR =", IMAGES_DIR)
print("DATA_ROOT  =", DATA_ROOT)

In [ ]:
def high_level_label(gz2_class):
    """Collapse detailed GZ2 codes (Sc2t, Ei, SBb2m, ...) to a few training buckets."""
    if not isinstance(gz2_class, str) or gz2_class == "A":
        return None
    if gz2_class.startswith("E"):
        return "elliptical"
    if gz2_class.startswith("SB"):
        return "spiral_barred"
    if gz2_class.startswith("S"):
        return "spiral"
    return None


def load_labeled_table(mapping_csv, labels_csv):
    mapping = pd.read_csv(mapping_csv)
    labels = pd.read_csv(labels_csv)
    if "dr7objid" in labels.columns:
        labels = labels.rename(columns={"dr7objid": "objid"})
    df = mapping.merge(labels[["objid", "gz2_class"]], on="objid", how="inner")
    df["label"] = df["gz2_class"].map(high_level_label)
    return df.dropna(subset=["label"]).reset_index(drop=True)


def _link_image(src, dst):
    if dst.exists():
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(Path(src).resolve(), dst)
    except OSError:
        import shutil
        shutil.copy2(src, dst)
    return True


def build_split_imagefolder_layout(images_dir, df, out_root, per_class=200,
                                   train_frac=0.70, val_frac=0.15, test_frac=0.15, seed=42):
    images_dir, out_root = Path(images_dir), Path(out_root)
    summary = {}
    for label in sorted(df["label"].unique()):
        rows = df[df["label"] == label].sample(frac=1, random_state=seed)
        if len(rows) > per_class:
            rows = rows.head(per_class)
        n = len(rows)
        n_train = int(train_frac * n); n_val = int(val_frac * n)
        splits = {"train": rows.iloc[:n_train],
                  "val": rows.iloc[n_train:n_train + n_val],
                  "test": rows.iloc[n_train + n_val:]}
        summary[label] = {}
        for split_name, split_rows in splits.items():
            linked = 0
            for _, row in split_rows.iterrows():
                src = images_dir / f"{int(row.asset_id)}.jpg"
                dst = out_root / split_name / label / f"{int(row.asset_id)}.jpg"
                if src.exists() and _link_image(src, dst):
                    linked += 1
            summary[label][split_name] = linked
    return summary

# Uncomment once the raw data is downloaded:
# df = load_labeled_table(RAW_ROOT / "gz2_filename_mapping.csv", RAW_ROOT / "gz2_hart16.csv")
# build_split_imagefolder_layout(IMAGES_DIR, df, DATA_ROOT, per_class=200)
# print("layout built")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

train_ds = ImageFolder(root=DATA_ROOT / "train", transform=transform)
val_ds   = ImageFolder(root=DATA_ROOT / "val",   transform=transform)
test_ds  = ImageFolder(root=DATA_ROOT / "test",  transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

num_classes = len(train_ds.classes)
print("classes:", train_ds.classes, "| num_classes:", num_classes)

## Step 1 - Define the CNN

Two `Conv -> ReLU -> Pool` blocks (`3 -> 16 -> 32` channels) shrink `64x64 -> 32x32 -> 16x16`, so the feature maps are `(B, 32, 16, 16)` and the flattened size is `32*16*16 = 8192`. The head maps that to logits. No softmax on the output - `CrossEntropyLoss` adds it.

In [ ]:
class GalaxyCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),   # (B, 16, 64, 64)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (B, 16, 32, 32)
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # (B, 32, 32, 32)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (B, 32, 16, 16)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                 # (B, 8192)
            nn.Linear(32 * 16 * 16, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),                  # (B, num_classes) logits
        )

    def forward(self, x):                                 # x: (B, 3, 64, 64)
        x = self.features(x)
        x = self.classifier(x)
        return x

## Step 2 - Instantiate and move to the device

In [ ]:
model = GalaxyCNN(num_classes=num_classes).to(device)
print(model)

## Step 3 - Forward-pass one real batch and count parameters

The output should be `(batch_size, num_classes)`. The convolutions cost only a few thousand weights; the first `Linear` dominates - but the whole CNN is still spatially aware, unlike the Week-2 MLP.

In [ ]:
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)
logits = model(images)
print("logits shape:", logits.shape)   # expect (B, num_classes)

total = sum(p.numel() for p in model.parameters())
print(f"total parameters: {total:,}")

## Step 4 - Loss, optimiser, and a starting-loss sanity check

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

start_loss = criterion(logits, labels).item()
print(f"untrained loss: {start_loss:.4f}  (expected near ln({num_classes}) = {math.log(num_classes):.4f})")

## Step 5 - The training loop

The five steps every batch: `zero_grad -> forward -> loss -> backward -> step`. We accumulate a batch-size-weighted running loss and store the per-epoch average.

In [ ]:
num_epochs = 8
train_losses = []

for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()              # 1
        outputs = model(inputs)            # 2
        loss = criterion(outputs, targets) # 3
        loss.backward()                    # 4
        optimizer.step()                   # 5
        running += loss.item() * inputs.size(0)
    epoch_loss = running / len(train_loader.dataset)
    train_losses.append(epoch_loss)
    print(f"Epoch {epoch+1:2d}/{num_epochs}  train loss: {epoch_loss:.4f}")

## Step 6 - Plot the loss curve

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(range(1, num_epochs + 1), train_losses, marker="o")
plt.xlabel("epoch"); plt.ylabel("training loss")
plt.title("GalaxyCNN training loss")
plt.grid(True, alpha=0.3)
plt.show()

## Step 7 (stretch) - Save Part 1's weights for Part 2

Saving the `state_dict` lets the Part-2 notebook load these weights instead of re-training. On Colab, save to Drive so they survive a runtime recycle. Full details in [`07-saving-and-loading-models.md`](../07-saving-and-loading-models.md).

In [ ]:
torch.save(model.state_dict(), "galaxy_model.pth")
print("saved galaxy_model.pth")

# To persist across Colab sessions, mount Drive and save there instead:
# from google.colab import drive; drive.mount('/content/drive')
# torch.save(model.state_dict(), '/content/drive/MyDrive/galaxy_model.pth')

## Reflection (example answers)

1. **Where the parameters live.** Almost all weights sit in the first `Linear(8192, 128)` of the head (~1.05M); the two conv layers together are only ~5k. Unlike the Week-2 MLP, the CNN's *first* layer is a cheap convolution that sees the whole image with shared kernels, so the model is far more parameter-efficient relative to the spatial work it does.
2. **The loss curve.** It should start near `ln(num_classes)` and fall steadily, with small wiggles. A flat curve from the start usually means a missing `zero_grad`, a tiny learning rate, or model/data on different devices; a `nan` usually means the LR is too high, labels aren't `long`, or inputs aren't normalised.
3. **What Part 2 adds.** Low *training* loss can just mean memorisation. Part 2 evaluates on held-out validation/test data, plots train-vs-val curves to spot overfitting, and builds a confusion matrix to see *which* galaxy types the model confuses.